<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/slm_llamaindex_practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Support-Desk Assistant — SLMs + LlamaIndex

**A practical for the session "Small Language Models & LlamaIndex".**

You are going to build a question-answering assistant over a small set of company
policy documents. Everything runs on **free, openly available models** — no API key,
no paid service, and it works on a plain Colab CPU runtime (a T4 GPU just makes it faster).

| Role | Model | Size |
|---|---|---|
| Generation (the SLM) | `Qwen/Qwen2.5-0.5B-Instruct` | 0.5B |
| Embeddings (search) | `BAAI/bge-small-en-v1.5` | 33M |
| Reranking | `cross-encoder/ms-marco-MiniLM-L-6-v2` | 22M |

**How to use this notebook**

1. Run every cell top to bottom, once, without changing anything.
2. Then go back to the three cells marked **🧪 EXPERIMENT** — that is where the learning happens.
3. Not a programmer? Read the text, run the cells, and look at the printed output.
   Every code cell is preceded by a plain-English explanation of what it does.

> **Tip:** in Colab choose *Runtime → Change runtime type → T4 GPU* if it is offered.
> On CPU everything still works; generation is just a few seconds per answer instead of under one.

---
## Step 1 — Install

This pulls in LlamaIndex plus the Hugging Face integrations. It takes 2–3 minutes
the first time. You only need to run it once per session.

In [ ]:
%pip install -q \
    "llama-index-core>=0.14,<0.15" \
    "llama-index-llms-huggingface>=0.6" \
    "llama-index-embeddings-huggingface>=0.6" \
    "llama-index-retrievers-bm25>=0.7" \
    "transformers>=4.44" "accelerate" "sentence-transformers>=3.0" \
    "bm25s" "PyStemmer" \
    "jedi"
# PyStemmer: provides access to efficient algorithms for calculating a “stemmed” form of a word.
# Jedi is a powerful static analysis, autocompletion, and refactoring library for Python


print("Install finished. If Colab asks you to restart the runtime, do it, then re-run from Step 2.")

Install finished. If Colab asks you to restart the runtime, do it, then re-run from Step 2.


In [ ]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

#PyTorch is an open-source, Python-based deep learning and machine learning framework
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

Running on: cuda


---
## Step 2 — Configure the models

`Settings` is LlamaIndex's global configuration object. Whatever you put here is
inherited by every component you build afterwards.

**This is the single most important cell in the notebook.** If you skip
`Settings.embed_model`, LlamaIndex silently falls back to OpenAI and you will get
a *"no API key found"* error later with no obvious cause.

Two models are being loaded:

* **the embedding model** — turns text into a list of 384 numbers ("coordinates of meaning")
  so we can search by meaning rather than by keyword;
* **the small language model** — reads the passages we retrieve and writes the answer.

`max_new_tokens=256` caps how long an answer can be. `do_sample=False` means greedy
decoding: the model always picks its most likely next word, which makes runs
reproducible and reduces invention — exactly what you want for factual Q&A.

The first run downloads roughly 1 GB of weights. Be patient; it is cached afterwards.

In [ ]:
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
LLM_MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"

Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL, device=DEVICE)

Settings.llm = HuggingFaceLLM(
    model_name=LLM_MODEL,
    tokenizer_name=LLM_MODEL,
    context_window=2048,          # must hold: retrieved chunks + question + answer
    max_new_tokens=256,
    generate_kwargs={"do_sample": False},
    is_chat_model=True,           # use the model's chat template
    device_map="auto",
)

# global default chunking - we will look at this properly in Step 4
Settings.node_parser = SentenceSplitter(chunk_size=128, chunk_overlap=16)

print("Embedding dimensions:", len(Settings.embed_model.get_text_embedding("hello")))
print("Models ready.")

Embedding dimensions: 384
Models ready.


### A quick sanity check

Before wiring anything together, ask the model a question it has no way of knowing.
It has never seen our company's policies, so whatever it says here is invented.

**This is the problem RAG solves.** Keep the output in mind — we will ask the exact
same question again at the end, with retrieval switched on.

In [ ]:
print(Settings.llm.complete("How long does Northwind Supply take to process a refund?"))

 I'm sorry, but I cannot provide an answer to your question as there is no specific information available about how long Northwind Supply takes to process refunds. The processing time for refunds can vary depending on the nature of the transaction and the specific circumstances surrounding it. It's best to contact Northwind Supply directly or check their website for the most up-to-date information on their refund processing times. Additionally, you may want to consider contacting them through their customer service hotline or email to get more personalized advice on the process.


---
## Step 3 — Create and load the documents

A real project would point at a folder of PDFs, a Notion export, or a database.
To keep this reproducible we write six short policy documents to disk and then load
them with `SimpleDirectoryReader`.

Notice the `file_metadata` function. It attaches a **category** and a **source** to every
document as it is loaded. Metadata is cheap to add now and expensive to add later —
you cannot filter, cite, or apply access control on a field you never stored.

In [ ]:
import os

os.makedirs("data", exist_ok=True)

# One blob, split on a marker - keeps the notebook self-contained and easy to edit.
CORPUS = '''
===refunds.md===
# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase date.
Proof of purchase is required, and the order number must be quoted on the request.
Items returned in a condition that prevents resale may be refused or refunded in part.
Digital gift cards are non-refundable once the redemption code has been revealed.

## Processing times
Approved refunds are issued to the original payment method within 5 business days.
For international orders, refunds take 10 business days because of cross-border
settlement times between the acquiring and issuing banks.
If the original payment method has been closed, we issue store credit instead, and
that credit appears on the account within 2 business days.

## Subscriptions
Annual subscription plans are refunded on a pro-rata basis for the unused months.
Monthly plans are not refunded for the current period, but they can be cancelled so
that no further charge is taken.
Refunds on subscriptions purchased through a reseller must be requested from that reseller.

===shipping.md===
# Shipping Policy

## Domestic delivery
Standard delivery arrives in 3 to 5 business days within the country.
Express delivery arrives the next business day for orders placed before 2:00 pm.
Orders placed after 2:00 pm on a Friday are dispatched on the following Monday.
Orders above 5,000 in value ship free of charge on the standard service.

## International delivery
International shipping takes 7 to 14 business days and may be delayed by customs.
Import duties and local taxes are the responsibility of the recipient.
We do not currently ship hardware to addresses in countries under trade restriction.

## Tracking and problems
A tracking number is emailed within 24 hours of dispatch.
If tracking has not updated for 5 business days, contact support and we will open a
carrier investigation, which normally concludes within 10 business days.

===warranty.md===
# Warranty Policy

## What is covered
All hardware carries a 24-month manufacturer warranty from the date of delivery.
The warranty covers manufacturing defects in materials and workmanship.
Accessories supplied in the box carry a 12-month warranty.

## What is not covered
The warranty does not cover accidental damage, liquid damage, damage caused by
unauthorised repair, or normal wear of consumable parts such as batteries and cables.
Cosmetic scratches that do not affect function are not covered.

## Making a claim
A warranty claim requires an RMA number, which support issues within one business day.
The RMA number must be written on the outside of the returned package.
Replacement units are dispatched within 7 business days of the faulty unit being received.
If the same unit fails twice within the warranty period we replace it with a new unit
rather than a repaired one.

===accounts.md===
# Account Access Policy

## Passwords
You can reset your password from the Settings page of your account.
A password reset link expires 60 minutes after it is issued.
Passwords must be at least 12 characters and cannot repeat any of your last 5 passwords.

## Two-factor authentication
Two-factor authentication can be enabled from Settings and is mandatory for admin users.
Recovery codes are shown once at setup and cannot be retrieved afterwards.
If you lose both your second factor and your recovery codes, identity verification with
support takes up to 2 business days.

## Lockouts
An account is locked after 5 consecutive failed sign-in attempts.
A locked account unlocks automatically after 30 minutes, or immediately if support
verifies your identity.
Repeated lockouts within 24 hours trigger a manual security review.

===privacy.md===
# Privacy Policy

## Where data is held
Customer data is stored in data centres located in the European Union.
Backups are encrypted at rest and are also held within the European Union.
We do not sell customer data to third parties under any circumstances.

## Retention
Account records are retained for 24 months after an account is closed.
Support conversations are retained for 12 months for quality and training purposes.
Billing records are retained for 7 years because tax law requires it, even after deletion
of the rest of the account.

## Your rights
You may request deletion of your personal data at any time; deletion completes within 30 days.
You may request an export of your data, which is delivered as a JSON archive within 14 days.
Requests are made from the Privacy tab in Settings, or by writing to the data protection team.

===business.md===
# Business Accounts Policy

## Eligibility and pricing
Business accounts are available to organisations with 5 or more seats.
A volume discount of 15 percent applies above 50 seats, and 25 percent above 200 seats.
Educational and non-profit organisations receive an additional 10 percent on top of the
volume discount, subject to verification.

## Billing
Business invoices are issued monthly on net-30 payment terms.
Purchase order numbers can be attached to invoices from the Billing tab.
Late payment beyond 45 days suspends new seat provisioning until the balance is cleared.

## Account management
Organisations above 200 seats are assigned a dedicated account manager.
Seat counts can be adjusted once per billing cycle, and reductions take effect at the
start of the next cycle rather than immediately.
Single sign-on is available on business plans at no extra cost.
'''

chunks = [c for c in CORPUS.split("===") if c.strip()]
POLICIES = {chunks[i].strip(): chunks[i + 1].strip() for i in range(0, len(chunks), 2)}

for name, body in POLICIES.items():
    with open(os.path.join("data", name), "w") as f:
        f.write(body + "\n")

print("Wrote", len(POLICIES), "policy files to ./data:", ", ".join(POLICIES))

Wrote 6 policy files to ./data: refunds.md, shipping.md, warranty.md, accounts.md, privacy.md, business.md


In [ ]:
from llama_index.core import SimpleDirectoryReader

CATEGORY = {
    "refunds.md":  "billing",
    "shipping.md": "shipping",
    "warranty.md": "warranty",
    "accounts.md": "account",
    "privacy.md":  "privacy",
    "business.md": "billing",
}

def file_metadata(path: str) -> dict:
    name = os.path.basename(path)
    return {"source": name, "category": CATEGORY.get(name, "general"), "version": "2026-08"}

docs = SimpleDirectoryReader("data", file_metadata=file_metadata).load_data()

# keep noisy auto-metadata out of the prompt (it wastes our tiny context window)
for d in docs:
    d.excluded_llm_metadata_keys   = ["file_path", "file_size", "creation_date",
                                      "last_modified_date", "file_type", "version"]
    d.excluded_embed_metadata_keys = list(d.excluded_llm_metadata_keys)

print(f"Loaded {len(docs)} documents\n")
for d in docs:
    print(f"  {d.metadata['source']:<14} category={d.metadata['category']:<9} {len(d.text)} chars")

Loaded 6 documents

  accounts.md    category=account   829 chars
  business.md    category=billing   874 chars
  privacy.md     category=privacy   843 chars
  refunds.md     category=billing   1054 chars
  shipping.md    category=shipping  850 chars
  warranty.md    category=warranty  885 chars


---
## Step 4 — Chunk into Nodes

A **Document** is a whole file. A **Node** is a chunk of that file, which is what
actually gets embedded and retrieved.

Why split at all? Two reasons:

1. An embedding squeezes a whole chunk into one point in meaning-space. A chunk about
   one idea gives a sharp, specific point. A chunk about eight ideas gives a blurry
   average that matches nothing well.
2. Our model has a 2048-token window. Everything we retrieve has to fit inside it,
   alongside the question and the answer.

`chunk_overlap` repeats a little text across the boundary so that an answer straddling
two chunks is intact in at least one of them.

> **Why 128 and not the 256–512 from the slides?** Our policy documents are unusually
> short — roughly 220 tokens each. At 512 every document would stay whole, and there
> would be nothing to demonstrate. 128 gives us section-sized chunks, which is what
> you would get from a real policy document of a few pages. Chunk size always depends
> on your documents; measure it rather than copying a number.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=128, chunk_overlap=16)
nodes = splitter.get_nodes_from_documents(docs)

print(f"{len(docs)} documents  ->  {len(nodes)} nodes\n")
print("Example node")
print("-" * 70)
print("metadata:", {k: v for k, v in nodes[0].metadata.items() if k in ("source", "category")})
print(nodes[0].text[:400], "...")

6 documents  ->  12 nodes

Example node
----------------------------------------------------------------------
metadata: {'source': 'accounts.md', 'category': 'account'}
# Account Access Policy

## Passwords
You can reset your password from the Settings page of your account.
A password reset link expires 60 minutes after it is issued.
Passwords must be at least 12 characters and cannot repeat any of your last 5 passwords.

## Two-factor authentication
Two-factor authentication can be enabled from Settings and is mandatory for admin users.
Recovery codes are shown  ...


### 🧪 EXPERIMENT 1 — chunk size

The cell below splits the same six documents at four different chunk sizes so you can
see the trade-off directly.

* Small chunks → many nodes, each very specific. Retrieval is precise but an answer
  that spans a few sentences can get cut in half.
* Large chunks → few nodes, each covering several topics. Retrieval gets vague,
  and each retrieved chunk eats more of the model's context window.

Then change `chunk_size` in the cell *above*, delete the `./storage` folder, re-run
Step 5, and look at how the retrieval scores in Step 6 move.

In [ ]:
for size in (96, 192, 384, 1024):
    n = SentenceSplitter(chunk_size=size, chunk_overlap=int(size * 0.12)).get_nodes_from_documents(docs)
    avg = sum(len(x.text) for x in n) / len(n)
    print(f"chunk_size={size:<5} -> {len(n):>3} nodes, average {avg:>6.0f} characters each")

chunk_size=96    ->  15 nodes, average    354 characters each
chunk_size=192   ->   7 nodes, average    775 characters each
chunk_size=384   ->   6 nodes, average    888 characters each
chunk_size=1024  ->   6 nodes, average    888 characters each


---
## Step 5 — Build and persist the index

`VectorStoreIndex` embeds every node and stores the vectors so they can be searched.
This is the **only slow step** in the whole pipeline — so we save it to disk with
`persist()`. On the next run it loads in about a second instead of re-embedding.

⚠️ If you later change the chunk size or the embedding model, **delete `./storage`**
and rebuild. Otherwise you will keep querying a stale index and get confusing results
with no error message.

In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage

PERSIST_DIR = "./storage"

if os.path.exists(PERSIST_DIR):
    index = load_index_from_storage(StorageContext.from_defaults(persist_dir=PERSIST_DIR))
    print("Loaded existing index from disk.")
else:
    index = VectorStoreIndex(nodes, show_progress=True)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print("Built the index and saved it to", PERSIST_DIR)

print("\nWhat persist() wrote:")
for f in sorted(os.listdir(PERSIST_DIR)):
    print("  ", f)

Loaded existing index from disk.

What persist() wrote:
   default__vector_store.json
   docstore.json
   graph_store.json
   image__vector_store.json
   index_store.json


---
## Step 6 — Retrieval: the part that decides everything

`similarity_top_k` is how many chunks we fetch. The scores are cosine similarity —
how close the question and the chunk are in meaning-space.

Look at the output carefully. Notice that the question *"How long do refunds take?"*
matches a passage that never uses the phrase "how long". That is semantic search
doing its job.

**Build this habit:** whenever an answer is wrong, print the retriever output *first*.
If the correct passage is not in this list, no prompt and no model can save the answer.

In [ ]:
def show(hits, title=""):
    if title:
        print(title)
        print("=" * 78)
    for i, h in enumerate(hits, 1):
        text = " ".join(h.text.split())
        print(f"{i}. score={h.score:.3f}  [{h.metadata['source']} / {h.metadata['category']}]")
        print(f"   {text[:150]}...")
    print()

retriever = index.as_retriever(similarity_top_k=4)
show(retriever.retrieve("How long do refunds take?"), "QUERY: How long do refunds take?")
show(retriever.retrieve("I am locked out of my account"), "QUERY: I am locked out of my account")

QUERY: How long do refunds take?
1. score=0.793  [refunds.md / billing]
   # Refund Policy ## Eligibility Customers may request a refund within 30 days of the original purchase date. Proof of purchase is required, and the ord...
2. score=0.753  [refunds.md / billing]
   If the original payment method has been closed, we issue store credit instead, and that credit appears on the account within 2 business days. ## Subsc...
3. score=0.667  [warranty.md / warranty]
   Cosmetic scratches that do not affect function are not covered. ## Making a claim A warranty claim requires an RMA number, which support issues within...
4. score=0.642  [privacy.md / privacy]
   ## Your rights You may request deletion of your personal data at any time; deletion completes within 30 days. You may request an export of your data, ...

QUERY: I am locked out of my account
1. score=0.748  [accounts.md / account]
   ## Lockouts An account is locked after 5 consecutive failed sign-in attempts. A locked account unloc

---
## Step 7 — Metadata filters

Filtering narrows the search space *before* the search happens. Two benefits:

* **Precision** — plausible-but-wrong matches from other categories disappear.
* **Access control** — in a real system you filter by what the current user is
  allowed to see, so the model physically cannot read a document it should not.
  That is a far stronger guarantee than instructing a model to keep a secret.

In [ ]:
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

billing_only = MetadataFilters(filters=[
    MetadataFilter(key="category", value="billing", operator=FilterOperator.EQ)
])

q = "How long does it take?"
show(index.as_retriever(similarity_top_k=3).retrieve(q),
     f"QUERY: {q}   (no filter - notice shipping and warranty leaking in)")
show(index.as_retriever(similarity_top_k=3, filters=billing_only).retrieve(q),
     f"QUERY: {q}   (filtered to category='billing')")

QUERY: How long does it take?   (no filter - notice shipping and warranty leaking in)
1. score=0.645  [privacy.md / privacy]
   ## Your rights You may request deletion of your personal data at any time; deletion completes within 30 days. You may request an export of your data, ...
2. score=0.617  [accounts.md / account]
   # Account Access Policy ## Passwords You can reset your password from the Settings page of your account. A password reset link expires 60 minutes afte...
3. score=0.606  [shipping.md / shipping]
   Import duties and local taxes are the responsibility of the recipient. We do not currently ship hardware to addresses in countries under trade restric...

QUERY: How long does it take?   (filtered to category='billing')
1. score=0.588  [refunds.md / billing]
   # Refund Policy ## Eligibility Customers may request a refund within 30 days of the original purchase date. Proof of purchase is required, and the ord...
2. score=0.558  [refunds.md / billing]
   If the original pay

---
## Step 8 — Hybrid retrieval (meaning + keywords)

Embeddings are excellent at paraphrase and hopeless at exact tokens. An RMA number
or an error code has no meaningful "neighbourhood" in meaning-space.

BM25 keyword search is the opposite: unbeatable on exact terms, blind to paraphrase.

`QueryFusionRetriever` runs both and merges the two ranked lists, so you get
both behaviours. We set `num_queries=1` to skip LLM-generated query variations —
that would cost extra calls to our slow local model.

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever

bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5)

hybrid = QueryFusionRetriever(
    [index.as_retriever(similarity_top_k=5), bm25],
    similarity_top_k=5,
    num_queries=1,              # 1 = no LLM query rewriting
    mode="reciprocal_rerank",
    use_async=False,
    verbose=False,
)

q = "RMA number"
show(index.as_retriever(similarity_top_k=3).retrieve(q), f"QUERY: {q}   (vector only)")
show(hybrid.retrieve(q)[:3],                              f"QUERY: {q}   (hybrid: vector + BM25)")

DEBUG:bm25s:Building index from IDs objects


QUERY: RMA number   (vector only)
1. score=0.680  [warranty.md / warranty]
   Cosmetic scratches that do not affect function are not covered. ## Making a claim A warranty claim requires an RMA number, which support issues within...
2. score=0.619  [shipping.md / shipping]
   Import duties and local taxes are the responsibility of the recipient. We do not currently ship hardware to addresses in countries under trade restric...
3. score=0.612  [warranty.md / warranty]
   # Warranty Policy ## What is covered All hardware carries a 24-month manufacturer warranty from the date of delivery. The warranty covers manufacturin...

QUERY: RMA number   (hybrid: vector + BM25)
1. score=0.033  [warranty.md / warranty]
   Cosmetic scratches that do not affect function are not covered. ## Making a claim A warranty claim requires an RMA number, which support issues within...
2. score=0.033  [shipping.md / shipping]
   Import duties and local taxes are the responsibility of the recipient. We do not curr

---
## Step 9 — Reranking: retrieve wide, keep narrow

The embedding search compares two vectors that were computed **separately**. Fast,
but approximate.

A **cross-encoder** reads the question and the passage *together* and scores the pair
directly. Far more accurate — and far too slow to run over a whole corpus. So we use
the cheap search to get 10 candidates, then let the cross-encoder pick the best 3.

This matters more with a small model than a large one: we can only afford to send
about three chunks, so those three had better be the right ones.

In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank, SimilarityPostprocessor

reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=3,
    device=DEVICE,
)

from llama_index.core import QueryBundle

q = "Can I get money back on a yearly plan?"
wide = index.as_retriever(similarity_top_k=10).retrieve(q)
show(wide[:5], f"QUERY: {q}   (top 5 of 10 before reranking)")
show(reranker.postprocess_nodes(wide, QueryBundle(q)), f"QUERY: {q}   (after cross-encoder reranking)")

QUERY: Can I get money back on a yearly plan?   (top 5 of 10 before reranking)
1. score=0.727  [refunds.md / billing]
   If the original payment method has been closed, we issue store credit instead, and that credit appears on the account within 2 business days. ## Subsc...
2. score=0.652  [refunds.md / billing]
   # Refund Policy ## Eligibility Customers may request a refund within 30 days of the original purchase date. Proof of purchase is required, and the ord...
3. score=0.606  [business.md / billing]
   # Business Accounts Policy ## Eligibility and pricing Business accounts are available to organisations with 5 or more seats. A volume discount of 15 p...
4. score=0.585  [privacy.md / privacy]
   # Privacy Policy ## Where data is held Customer data is stored in data centres located in the European Union. Backups are encrypted at rest and are al...
5. score=0.574  [warranty.md / warranty]
   Cosmetic scratches that do not affect function are not covered. ## Making a claim A warranty

---
## Step 10 — The query engine: retrieval + a grounded prompt

Now we join everything: retrieve wide → rerank to 3 → hand those to the small model
with a strict prompt.

Two things in that prompt do the heavy lifting:

1. **"Use only the context"** — grounds the model in the retrieved text.
2. **"If the answer is not there, say you do not know"** — gives it a sanctioned way to
   fail. Without an exit, a model with nothing to say will invent something instead.

Small models follow short, literal instructions much better than long nuanced ones.
Keep the prompt boring.

In [ ]:
from llama_index.core.prompts import PromptTemplate

QA_PROMPT = PromptTemplate(
    "You are a customer support assistant for Northwind Supply.\n"
    "Answer the question using ONLY the context below.\n"
    "If the context does not contain the answer, reply exactly: "
    "\"I don't know based on the policy documents.\"\n"
    "Answer in one or two sentences.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer: "
)

engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[reranker],
    response_mode="compact",
    text_qa_template=QA_PROMPT,
)

def ask(question):
    r = engine.query(question)
    print("Q:", question)
    print("A:", str(r).strip())
    print("   sources:", ", ".join(
        f"{n.metadata['source']}({n.score:.2f})" for n in r.source_nodes))
    print("-" * 78)
    return r

_ = ask("How long do refunds take?")
_ = ask("What happens if I fail to sign in five times?")
_ = ask("Is accidental damage covered by the warranty?")
_ = ask("What discount applies above 200 seats?")

Q: How long do refunds take?
A: Refunds take 10 business days for international orders.
   sources: refunds.md(6.33), refunds.md(3.46), accounts.md(-4.86)
------------------------------------------------------------------------------
Q: What happens if I fail to sign in five times?
A: If you fail to sign in five times, your account will be locked.
   sources: accounts.md(1.38), warranty.md(-10.12), business.md(-10.95)
------------------------------------------------------------------------------
Q: Is accidental damage covered by the warranty?
A: Yes, accidental damage is covered under the warranty.
   sources: warranty.md(7.37), warranty.md(-2.91), refunds.md(-11.34)
------------------------------------------------------------------------------
Q: What discount applies above 200 seats?
A: The discount applies above 200 seats, specifically 25%.
   sources: business.md(7.10), business.md(1.55), shipping.md(-11.34)
-------------------------------------------------------------------------

### 🧪 EXPERIMENT 2 — the honesty test

Ask something the documents genuinely do not cover. A well-grounded system should
decline. An ungrounded one will invent a confident, plausible, wrong answer.

Run the cell, then **comment out `text_qa_template=QA_PROMPT`** in the cell above,
re-run both, and compare. This is the clearest demonstration in the notebook of why
prompt grounding matters more for small models, not less.

In [ ]:
_ = ask("Do you accept payment in cryptocurrency?")
_ = ask("What is the CEO's home address?")
_ = ask("How long does Northwind Supply take to process a refund?")   # compare with Step 2

Q: Do you accept payment in cryptocurrency?
A: No, I don't know based on the policy documents.
   sources: refunds.md(-11.09), refunds.md(-11.13), business.md(-11.30)
------------------------------------------------------------------------------
Q: What is the CEO's home address?
A: I don't know based on the policy documents.
   sources: shipping.md(-11.40), refunds.md(-11.40), business.md(-11.41)
------------------------------------------------------------------------------
Q: How long does Northwind Supply take to process a refund?
A: Northwind Supply processes refunds within 5 business days after receiving the proof of purchase and the order number.
   sources: refunds.md(-0.52), refunds.md(-4.06), privacy.md(-8.81)
------------------------------------------------------------------------------


---
## Step 11 — Memory: making it a conversation

A query engine answers one question and forgets. Ask *"and for international orders?"*
on its own and it retrieves documents about international shipping — the follow-up has
lost its subject.

`chat_mode="condense_plus_context"` fixes this in two steps:

1. rewrite the follow-up into a standalone question using the chat history;
2. retrieve using that *rewritten* question.

So memory improves **retrieval**, not just fluency.

Keep `token_limit` modest. Chat history and retrieved chunks compete for the same
2048-token window — every token spent on history is a token not available for evidence.

In [ ]:
from llama_index.core.memory import Memory

memory = Memory.from_defaults(session_id="workshop-user", token_limit=1200)

chat = index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=memory,
    similarity_top_k=4,
    node_postprocessors=[reranker],
    system_prompt=(
        "You are a customer support assistant for Northwind Supply. "
        "Answer only from the retrieved policy text. Be brief. "
        "If the policy does not cover it, say you don't know."
    ),
)

for turn in [
    "How long do refunds take?",
    "And for international orders?",     # only makes sense with memory
    "What about the shipping time for those?",
]:
    print("USER :", turn)
    print("BOT  :", str(chat.chat(turn)).strip())
    print("-" * 78)

USER : How long do refunds take?
BOT  : Refunds are issued to the original payment method within 5 business days. For international orders, refunds take 10 business days due to cross-border settlement times between the acquiring and issuing banks.
------------------------------------------------------------------------------
USER : And for international orders?
BOT  : Orders placed after 2:00 pm on a Friday are dispatched on the following Monday. Orders above 5,000 in value ship free of charge on the standard service.
------------------------------------------------------------------------------
USER : What about the shipping time for those?
BOT  : The shipping time for domestic deliveries is 3 to 5 business days, while express deliveries arrive the next business day for orders placed before 2:00 pm. Orders placed after 2:00 pm on a Friday are dispatched on the following Monday. Orders above 5,000 in value ship free of charge on the standard service.
-----------------------------------

In [ ]:
# what the memory is actually holding
for m in memory.get():
    content = " ".join(str(m.content).split())
    print(f"{str(m.role):<20} {content[:110]}")

# chat.reset()   # uncomment to clear the conversation and start again

MessageRole.USER     How long do refunds take?
MessageRole.ASSISTANT Refunds are issued to the original payment method within 5 business days. For international orders, refunds ta
MessageRole.USER     And for international orders?
MessageRole.ASSISTANT Orders placed after 2:00 pm on a Friday are dispatched on the following Monday. Orders above 5,000 in value sh
MessageRole.USER     What about the shipping time for those?
MessageRole.ASSISTANT The shipping time for domestic deliveries is 3 to 5 business days, while express deliveries arrive the next bu


---
## Step 12 — A tiny evaluation set

"It felt better" is not a measurement. Twenty questions with known-correct sources
turns tuning from an argument into an experiment.

Here we measure **hit rate**: how often the correct document appears in the retrieved
results. This is a *retrieval* metric — it says nothing about the wording of the answer,
which is exactly why it is useful. Retrieval and generation should be measured separately.

In [ ]:
EVAL = [
    ("How long do refunds take?",                    "refunds.md"),
    ("Can I return something after six weeks?",      "refunds.md"),
    ("When will my parcel arrive?",                  "shipping.md"),
    ("Do you ship abroad?",                          "shipping.md"),
    ("Is water damage covered?",                     "warranty.md"),
    ("How do I start a warranty claim?",             "warranty.md"),
    ("I forgot my password",                         "accounts.md"),
    ("How long is my data kept?",                    "privacy.md"),
    ("Do you sell my information?",                  "privacy.md"),
    ("What is the bulk pricing for 100 users?",      "business.md"),
]

def hit_rate(retrieve_fn, k):
    hits = 0
    misses = []
    for question, expected in EVAL:
        got = [n.metadata["source"] for n in retrieve_fn(question)[:k]]
        if expected in got:
            hits += 1
        else:
            misses.append((question, expected, got))
    return hits / len(EVAL), misses

from llama_index.core import QueryBundle

plain  = lambda q: index.as_retriever(similarity_top_k=3).retrieve(q)
reranked = lambda q: reranker.postprocess_nodes(
    index.as_retriever(similarity_top_k=10).retrieve(q), QueryBundle(q))

for label, fn in [("vector top-3", plain), ("top-10 then reranked to 3", reranked)]:
    score, misses = hit_rate(fn, 3)
    print(f"{label:<28} hit rate @3 = {score:.0%}")
    for q, exp, got in misses:
        print(f"    MISS  {q!r}  expected {exp}, got {got}")
print()
print("Add your own questions to EVAL and re-run before changing anything else.")

vector top-3                 hit rate @3 = 100%
top-10 then reranked to 3    hit rate @3 = 100%

Add your own questions to EVAL and re-run before changing anything else.


### 🧪 EXPERIMENT 3 — make a change, then measure it

Pick one change and measure its effect on the hit rate above. Change **one thing at a time**.

* Set `chunk_size=96` in Step 4, delete `./storage`, rebuild the index in Step 5, re-run this cell.
* Drop `similarity_top_k` to 1 and see which questions start failing.
* Swap the embedding model to `BAAI/bge-base-en-v1.5` (larger, slower, usually better)
  — remember to delete `./storage` first.
* Add the hybrid retriever from Step 8 as a third row in the comparison.

---
## Where to take this next

**Use your own documents.** Drop PDFs or Markdown files into `./data`, delete `./storage`,
and re-run from Step 3. `SimpleDirectoryReader` handles PDF, DOCX, TXT and Markdown out of the box.

**Try a bigger model.** Change `LLM_MODEL` to `Qwen/Qwen3-1.7B` or
`meta-llama/Llama-3.2-1B-Instruct` (the Llama licence must be accepted on Hugging Face first).
Raise `context_window` to 4096 and you can afford more retrieved chunks.

**Persist the memory.** `Memory.from_defaults(..., async_database_uri="sqlite+aiosqlite:///memory.db")`
keeps conversations across restarts, keyed by `session_id`.

**Add real evaluation.** LlamaIndex ships `FaithfulnessEvaluator` (is the answer supported
by the retrieved text?) and `RelevancyEvaluator` (did it address the question?) to sit
alongside the hit rate you measured above.

---

### The three things worth remembering

1. **A small model is a good reader and a poor encyclopaedia.** Do not ask it what it knows —
   give it what it needs to read.
2. **Retrieval quality is the ceiling on answer quality.** Debug the retriever before the prompt.
3. **Chunking and reranking beat model size.** Time spent there pays back more than
   time spent shopping for a bigger model.